In [1]:
import pandas as pd 
import numpy as np
import tensorflow as tf 
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers,models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
import keras_tuner as kt

In [2]:
df_train = pd.read_csv(r'../data/selected_col/model_train.csv')
df_test = pd.read_csv(r'../data/selected_col/model_test.csv')
df_val = pd.read_csv(r'../data/selected_col/model_val.csv')

In [3]:
img_size = (224,224)
batch_size = 32

In [4]:
batch_size_64 = 64

In [5]:
data_augmentation_2 = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.2),
])

In [6]:
def preprocess_image2(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    return image, label

In [7]:
def preprocess_train2(image_path, label):
    image, label = preprocess_image2(image_path, label)
    image = data_augmentation_2(image)
    return image, label

In [8]:
def preprocess_test2(image_path, label):
    image, label = preprocess_image2(image_path, label)
    return image, label

In [9]:
train_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset2 = (
    train_dataset2
    .map(preprocess_train2, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [10]:
test_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset2 = (
    test_dataset2
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [11]:
val_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset2 = (
    val_dataset2
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


In [ ]:
#dataset 2 64

In [68]:
train_dataset2_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset2_64 = (
    train_dataset2_64
    .map(preprocess_train2, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [70]:
test_dataset2_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset2_64 = (
    test_dataset2_64
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [71]:
val_dataset2_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset2_64 = (
    val_dataset2_64
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


In [19]:
y_train = df_train["dx_encode"].values


class_weights = compute_class_weight(class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train)

# Convert to dictionary
class_weight_dict = dict(enumerate(class_weights))

class_weight_dict

{0: np.float64(4.372426699937617),
 1: np.float64(2.7813492063492062),
 2: np.float64(1.3020620471855842),
 3: np.float64(12.51607142857143),
 4: np.float64(1.2853475151292866),
 5: np.float64(0.2133572798392743),
 6: np.float64(10.113997113997113)}

In [12]:
'''CNN BASELINE WITH EARLY STOP'''

early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [13]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
'eraly stop'

In [19]:
ef_base_model3 = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

ef_base_model3.trainable = False

efficientnet_model3 = models.Sequential([

    ef_base_model3,

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),

    layers.Dense(7, activation="softmax")

])

efficientnet_model3.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [22]:
efficientnet_model3.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [23]:
history = efficientnet_model3.fit(train_dataset2,validation_data=val_dataset2,epochs=10,class_weight=class_weight_dict,callbacks=[early_stop
                                                                                                                                  
                                                                                                                                  ])

Epoch 1/10


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 149s 627ms/step - accuracy: 0.4383 - loss: 1.4763 - val_accuracy: 0.5602 - val_loss: 1.1022
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 616s 3s/step - accuracy: 0.5551 - loss: 1.1384 - val_accuracy: 0.5675 - val_loss: 1.0471
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 124s 556ms/step - accuracy: 0.5957 - loss: 0.9881 - val_accuracy: 0.6374 - val_loss: 0.9306
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 404s 2s/step - accuracy: 0.6094 - loss: 0.9949 - val_accuracy: 0.6128 - val_loss: 0.9917
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 1002s 5s/step - accuracy: 0.6336 - loss: 0.9052 - val_accuracy: 0.6627 - val_loss: 0.8744
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 121s 545ms/step - accuracy: 0.6388 - loss: 0.8763 - val_accuracy: 0.6474 - val_loss: 0.8884
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 3557s 16s/step - accuracy: 0.6603 - loss: 0.8143 - val_accuracy: 0.6720 - val_loss: 0.8681
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 136s 611ms/step - accuracy: 0.6696 - loss: 0.7874 - val_

In [24]:
efficientnet_model3.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 21s 448ms/step - accuracy: 0.6534 - loss: 0.8822


[0.8821983337402344, 0.6533599495887756]

In [26]:
efficientnet_model3.save(r'../models/efficient_early_stop.keras')

In [27]:
history_df = pd.DataFrame(history.history)


history_df.to_csv(r'../log/efficient_early_history.csv', index=False)

In [ ]:
#learning rate

In [29]:
ef_base_model3 = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

ef_base_model3.trainable = False

efficientnet_model_lr = models.Sequential([

    ef_base_model3,

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),

    layers.Dense(7, activation="softmax")

])

efficientnet_model_lr.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [30]:
efficientnet_model_lr.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [31]:
history = efficientnet_model_lr.fit(train_dataset2,validation_data=val_dataset2,epochs=5,class_weight=class_weight_dict,callbacks=[lr_scheduler])

Epoch 1/5


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 146s 611ms/step - accuracy: 0.4518 - loss: 1.4446 - val_accuracy: 0.5722 - val_loss: 1.1165 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 657s 3s/step - accuracy: 0.5644 - loss: 1.1338 - val_accuracy: 0.6148 - val_loss: 0.9752 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 207s 935ms/step - accuracy: 0.6034 - loss: 1.0123 - val_accuracy: 0.5995 - val_loss: 1.0515 - learning_rate: 0.0010
Epoch 4/5
219/220 ━━━━━━━━━━━━━━━━━━━━ 0s 501ms/step - accuracy: 0.6069 - loss: 0.9764
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 601ms/step - accuracy: 0.6069 - loss: 0.9764 - val_accuracy: 0.5383 - val_loss: 1.2404 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 600ms/step - accuracy: 0.6395 - loss: 0.8789 - val_accuracy: 0.5875 - val_loss: 1.0477 - learning_rate: 5.0000e-04


In [32]:
test_loss,test_accuracy = efficientnet_model_lr.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 25s 519ms/step - accuracy: 0.5855 - loss: 1.0610


In [33]:
efficientnet_model_lr.save(r'../models/effecient_learning_rate.keras')

In [ ]:
'optimizer'

In [ ]:
'adam'

In [37]:
ef_base_model3 = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

ef_base_model3.trainable = False

efficientnet_model_adam = models.Sequential([

    ef_base_model3,

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),

    layers.Dense(7, activation="softmax")

])

efficientnet_model_adam.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [38]:
efficientnet_model_adam.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [39]:
history = efficientnet_model_adam.fit(train_dataset2,validation_data=val_dataset2,epochs=5,class_weight=class_weight_dict)

Epoch 1/5


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 146s 586ms/step - accuracy: 0.4563 - loss: 1.4700 - val_accuracy: 0.5782 - val_loss: 1.0652
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 147s 661ms/step - accuracy: 0.5589 - loss: 1.1435 - val_accuracy: 0.6061 - val_loss: 1.0543
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 128s 577ms/step - accuracy: 0.5758 - loss: 1.0427 - val_accuracy: 0.5549 - val_loss: 1.1072
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 618s 3s/step - accuracy: 0.6206 - loss: 0.9512 - val_accuracy: 0.5163 - val_loss: 1.2462
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 149s 664ms/step - accuracy: 0.6338 - loss: 0.8849 - val_accuracy: 0.6959 - val_loss: 0.7970


In [40]:
test_loss,test_accuracy = efficientnet_model_adam.evaluate(test_dataset2)

print(f"""test_loss:{test_loss}
      test accuracy: {test_accuracy}""" )

47/47 ━━━━━━━━━━━━━━━━━━━━ 24s 500ms/step - accuracy: 0.6680 - loss: 0.8232
test_loss:0.8231519460678101
      test accuracy: 0.6679973602294922


In [41]:
efficientnet_model_adam.save(r'../models/effecient_adam.keras')

In [42]:
history_df = pd.DataFrame(history.history)


history_df.to_csv(r'../log/efficient_adam_history.csv', index=False)

In [ ]:
#sgd

In [43]:
ef_base_model3 = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

ef_base_model3.trainable = False

efficientnet_model_sgd = models.Sequential([

    ef_base_model3,

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),

    layers.Dense(7, activation="softmax")

])

efficientnet_model_sgd.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [45]:
efficientnet_model_sgd.compile(optimizer="sgd",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [46]:
history = efficientnet_model_sgd.fit(train_dataset2,validation_data=val_dataset2,epochs=5,class_weight=class_weight_dict)

Epoch 1/5


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 154s 628ms/step - accuracy: 0.3441 - loss: 1.6870 - val_accuracy: 0.4072 - val_loss: 1.4913
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 138s 620ms/step - accuracy: 0.4841 - loss: 1.3792 - val_accuracy: 0.5915 - val_loss: 1.1131
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 130s 586ms/step - accuracy: 0.5252 - loss: 1.2436 - val_accuracy: 0.6128 - val_loss: 1.0501
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1198s 5s/step - accuracy: 0.5560 - loss: 1.1624 - val_accuracy: 0.6327 - val_loss: 0.9769
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 130s 586ms/step - accuracy: 0.5698 - loss: 1.1243 - val_accuracy: 0.6467 - val_loss: 0.9942


In [47]:
test_loss,test_accuracy = efficientnet_model_sgd.evaluate(test_dataset2)

print(f"""
      test loss : {test_loss}
      test_accuracy  : {test_accuracy}
      """)

47/47 ━━━━━━━━━━━━━━━━━━━━ 26s 555ms/step - accuracy: 0.6068 - loss: 1.0139

      test loss : 1.0138550996780396
      test_accuracy  : 0.6067864298820496
      


In [49]:
efficientnet_model_sgd.save(r'../models/efficient_sgd.keras')

In [ ]:
'''rms'''

In [50]:
ef_base_model3 = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

ef_base_model3.trainable = False

efficientnet_model_rms = models.Sequential([

    ef_base_model3,

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),

    layers.Dense(7, activation="softmax")

])

efficientnet_model_rms.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [52]:
efficientnet_model_rms.compile(optimizer=tf.keras.optimizers.RMSprop(),loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [53]:
history = efficientnet_model_rms.fit(train_dataset2,validation_data=val_dataset2,epochs=5,class_weight=class_weight_dict)

Epoch 1/5


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 141s 597ms/step - accuracy: 0.4745 - loss: 1.4755 - val_accuracy: 0.6600 - val_loss: 0.9139
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 323s 1s/step - accuracy: 0.5556 - loss: 1.1637 - val_accuracy: 0.5536 - val_loss: 1.1411
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 129s 580ms/step - accuracy: 0.5895 - loss: 1.0735 - val_accuracy: 0.6347 - val_loss: 0.9854
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 133s 599ms/step - accuracy: 0.6035 - loss: 0.9865 - val_accuracy: 0.5376 - val_loss: 1.2532
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 133s 595ms/step - accuracy: 0.6172 - loss: 0.9790 - val_accuracy: 0.6913 - val_loss: 0.8220


In [79]:
efficientnet_model_rms.save(r'../models/efficient_rms.keras')

In [ ]:
'''64'''

In [73]:
ef_base_model3 = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

ef_base_model3.trainable = False

efficientnet_model_64 = models.Sequential([

    ef_base_model3,

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),

    layers.Dense(7, activation="softmax")

])

efficientnet_model_64.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_7      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [75]:
efficientnet_model_64.compile(optimizer='adam',loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [77]:
history = efficientnet_model_64.fit(train_dataset2_64,validation_data=val_dataset2_64,epochs=5,class_weight=class_weight_dict,callbacks=[early_stop])

Epoch 1/5


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


110/110 ━━━━━━━━━━━━━━━━━━━━ 171s 1s/step - accuracy: 0.4429 - loss: 1.4614 - val_accuracy: 0.5622 - val_loss: 1.1336
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 154s 1s/step - accuracy: 0.5717 - loss: 1.1136 - val_accuracy: 0.5010 - val_loss: 1.2645
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 173s 2s/step - accuracy: 0.5905 - loss: 1.0368 - val_accuracy: 0.6387 - val_loss: 0.9375
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 142s 1s/step - accuracy: 0.6148 - loss: 0.9630 - val_accuracy: 0.6647 - val_loss: 0.8916
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 145s 1s/step - accuracy: 0.6389 - loss: 0.8790 - val_accuracy: 0.6341 - val_loss: 0.9473
Restoring model weights from the end of the best epoch: 4.


In [78]:
test_loss,test_accuracy = efficientnet_model_64.evaluate(test_dataset2_64)

24/24 ━━━━━━━━━━━━━━━━━━━━ 24s 990ms/step - accuracy: 0.6434 - loss: 0.9057


In [80]:
efficientnet_model_64.save(r'../models/effecient_64.keras')

In [ ]:
'''hyper parameter'''

In [81]:
NUM_CLASSES =  7
def build_efficientnet(hp):
    
    
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(224,224,3)
    )

    base_model.trainable = False

    model = tf.keras.Sequential([

        base_model,

        tf.keras.layers.GlobalAveragePooling2D(),

        tf.keras.layers.Dense(
            hp.Choice(
                "dense_units",
                values=[128,256,512]
            ),
            activation="relu"
        ),

        tf.keras.layers.Dropout(
            hp.Choice(
                "dropout",
                values=[0.2,0.3,0.5]
            )
        ),

        tf.keras.layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )

    ])

    model.compile(
        optimizer=hp.Choice(
        "optimizer",
        ["adam","rmsprop","sgd"]
    ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [82]:
tuner = kt.RandomSearch(
    build_efficientnet,
    objective="val_accuracy",
    max_trials=5,
    overwrite=True,
    directory="hyperparameter_tuning",
    project_name="effecient"
)

In [83]:
tuner.search(
    train_dataset2,
    validation_data=val_dataset2,
    epochs=5,
    class_weight=class_weight_dict
)

Trial 5 Complete [00h 11m 19s]
val_accuracy: 0.6513639092445374

Best val_accuracy So Far: 0.7312042713165283
Total elapsed time: 01h 23m 46s


In [84]:
best_hps = tuner.get_best_hyperparameters(1)[0]

print(best_hps.values)

{'dense_units': 512, 'dropout': 0.3, 'optimizer': 'rmsprop'}


In [85]:
best_model = tuner.get_best_models(1)[0]

c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(store)


In [86]:
history = best_model.fit(
    train_dataset2,
    validation_data=val_dataset2,
    epochs=5,
    class_weight=class_weight_dict,
    callbacks=[early_stop]
)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 153s 648ms/step - accuracy: 0.6151 - loss: 1.0529 - val_accuracy: 0.7212 - val_loss: 0.7812
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 168s 758ms/step - accuracy: 0.6132 - loss: 1.0534 - val_accuracy: 0.6653 - val_loss: 0.9568
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 376s 2s/step - accuracy: 0.6305 - loss: 1.0134 - val_accuracy: 0.6933 - val_loss: 0.8558
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 138s 618ms/step - accuracy: 0.6275 - loss: 1.0116 - val_accuracy: 0.6806 - val_loss: 0.9150
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [87]:
test_loss, test_accuracy = best_model.evaluate(test_dataset2)

print("Test Accuracy :", test_accuracy)
print("Test Loss :", test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 24s 518ms/step - accuracy: 0.7139 - loss: 0.8001
Test Accuracy : 0.7139055132865906
Test Loss : 0.8000591993331909


In [88]:
best_model.save(r'../models/hyper_effecient.keras')

In [89]:
history_df = pd.DataFrame(history.history)


history_df.to_csv(r'../log/efficient_hyper_history.csv', index=False)

In [15]:
ef_base_mode = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

ef_base_mode.trainable = False

efficientnet = models.Sequential([

    ef_base_mode,

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),

    layers.Dense(7, activation="softmax")

])

efficientnet.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [16]:
efficientnet.compile(optimizer='adam',loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [20]:
history = efficientnet.fit(train_dataset2,validation_data=val_dataset2,epochs=5,class_weight=class_weight_dict,callbacks=[early_stop])

Epoch 1/5


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 160s 652ms/step - accuracy: 0.4524 - loss: 1.4699 - val_accuracy: 0.6314 - val_loss: 0.9827
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 144s 647ms/step - accuracy: 0.5557 - loss: 1.1491 - val_accuracy: 0.6314 - val_loss: 0.9434
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 148s 665ms/step - accuracy: 0.5814 - loss: 1.0490 - val_accuracy: 0.5709 - val_loss: 1.1070
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 183s 817ms/step - accuracy: 0.6162 - loss: 0.9456 - val_accuracy: 0.6401 - val_loss: 0.9373
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 148s 658ms/step - accuracy: 0.6395 - loss: 0.8988 - val_accuracy: 0.5522 - val_loss: 1.1269
Restoring model weights from the end of the best epoch: 4.


In [21]:
base_model = efficientnet.layers[0]
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

efficientnet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


history_densenet_ft = efficientnet.fit(
    train_dataset2,
    validation_data=val_dataset2,
    epochs=5,
    class_weight=class_weight_dict,
    callbacks=[early_stop, lr_scheduler])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 189s 764ms/step - accuracy: 0.5424 - loss: 1.6428 - val_accuracy: 0.4717 - val_loss: 1.4227 - learning_rate: 1.0000e-05
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 172s 772ms/step - accuracy: 0.5289 - loss: 1.3303 - val_accuracy: 0.4731 - val_loss: 1.3915 - learning_rate: 1.0000e-05
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 156s 702ms/step - accuracy: 0.5275 - loss: 1.1621 - val_accuracy: 0.4824 - val_loss: 1.3607 - learning_rate: 1.0000e-05
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 191s 860ms/step - accuracy: 0.5362 - loss: 1.1160 - val_accuracy: 0.4983 - val_loss: 1.3180 - learning_rate: 1.0000e-05
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 154s 692ms/step - accuracy: 0.5470 - loss: 1.0492 - val_accuracy: 0.5063 - val_loss: 1.2880 - learning_rate: 1.0000e-05
Restoring model weights from the end of the best epoch: 5.


In [22]:
efficientnet.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 23s 493ms/step - accuracy: 0.5057 - loss: 1.3009


[1.3009084463119507, 0.5056553483009338]

In [23]:
efficientnet.save(r'../models/fine_tune_effiecient.keras')